# PINNsFormer + Chebyshev-4: Navier–Stokes × 3 Seeds

This notebook runs the 4-objective Chebyshev-center method from Yoon et al. (2026), arXiv:2605.09975, with `p=4`.

Objectives: `L_u_data`, `L_v_data`, `L_f_u`, `L_f_v`. Fair-comparison defaults match the current HARMONIC/M-ConFIG runners: 800 training points, pseudo sequence length 5, seeds 0/1/2, 1000 Adam steps, and `lr=1e-4`.

The reusable implementation is in `chebyshev_runner.py`: p-norm gradient normalization -> simplex dual Frank-Wolfe solve -> primal direction recovery -> adaptive scalar -> Adam update. Raw outputs are written under `pinnsformer-config/outputs/navier_stokes_chebyshev4/`.


In [ ]:
from pathlib import Path
import os, sys

PINNSFORMER_ROOT = Path(os.environ.get('PINNSFORMER_ROOT', '/home/simplexity/cyt/pinnsformer-main')).resolve()
EXPERIMENT_ROOT = Path(os.environ.get('PINNSFORMER_CONFIG_ROOT', str(PINNSFORMER_ROOT / 'pinnsformer-config'))).resolve()
CHEB_DIR = EXPERIMENT_ROOT / 'notebooks' / 'navier_stokes' / 'chebyshev'
assert EXPERIMENT_ROOT.exists(), EXPERIMENT_ROOT
assert (CHEB_DIR / 'chebyshev_runner.py').exists(), CHEB_DIR / 'chebyshev_runner.py'
if str(CHEB_DIR) not in sys.path:
    sys.path.insert(0, str(CHEB_DIR))
from chebyshev_runner import ExperimentConfig, run_multiseed
print('PINNSFORMER_ROOT:', PINNSFORMER_ROOT)
print('EXPERIMENT_ROOT:', EXPERIMENT_ROOT)
print('CHEB_DIR:', CHEB_DIR)


In [ ]:
cfg = ExperimentConfig(
    p=4, seeds=(0, 1, 2), epochs=1000, lr=1e-4,
    fw_max_iters=100, fw_tol=1e-6, stop_tol=1e-6, force_rerun=False,
)
summary = run_multiseed(cfg)
summary


## Notes

- `p=2` is the primary Chebyshev configuration for the current comparison.
- `p=4` is the required norm-geometry ablation.
- Do not promote output CSVs into `results/summaries/` until the real server runs finish and the numbers are checked.
- Existing per-seed `metrics.csv` files are reused unless `force_rerun=True`.
